In [1]:
import os, re, json, math, random, gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


In [2]:
posts_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\posts_df.csv')
comments_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\comments_df.csv')
image_description = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\src\Image_Description\fashion_description.csv')

In [4]:
if 'timestamp' in posts_df.columns:
    posts_df['timestamp'] = pd.to_datetime(posts_df['timestamp'], errors='coerce')
if 'timestamp' in comments_df.columns:
    comments_df['timestamp'] = pd.to_datetime(comments_df['timestamp'], errors='coerce')

print(f"posts_df shape: {posts_df.shape}")
print("Columns:", list(posts_df.columns)[:20])
print("---------------------------")
print(f"comments_df shape: {comments_df.shape}")
print("comments_df:", comments_df.shape, "| columns:", list(comments_df.columns))

print(f"image_description shape: {image_description.shape}")
print("image_description:", list(image_description.columns))

posts_df shape: (827, 6)
Columns: ['post_id', 'postUser', 'timestamp', 'likesCount', 'commentsCount', 'keywords_posts']
---------------------------
comments_df shape: (5361, 7)
comments_df: (5361, 7) | columns: ['post_id', 'commentUser', 'timestamp', 'likes', 'polarity', 'sentiment', 'keywords_comments']
image_description shape: (822, 7)
image_description: ['post_id', 'description', 'color', 'category', 'style', 'occasion', 'keywords']


In [7]:
def prepare_user_sequences(posts_df, comments_df):
    user_interactions = comments_df.merge(
        posts_df[['post_id', 'keywords_posts']], 
        on='post_id', 
        how='left'
    )
    
    # Sort timestamp
    user_interactions = user_interactions.sort_values('timestamp')
    # Drop columns "likes" 
    user_interactions = user_interactions.drop(columns=['likes'])
    
    return user_interactions

user_interactions = prepare_user_sequences(posts_df, comments_df)
user_interactions.head()

,post_id,commentUser,timestamp,polarity,sentiment,keywords_comments,keywords_posts
5331,822,alantrowe,2021-01-04 10:38:18+00:00,0.2468,neutral,['green_heart'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ..."
5332,823,alantrowe,2021-01-04 10:39:08+00:00,0.0711,neutral,['heart_suit'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ..."
5330,822,thedramatiara,2021-01-05 20:29:21+00:00,-0.0089,neutral,['heart_with_arrow'],"['prada', 'fashion', 'fw', 'vogue', 'italia', ..."
5299,811,wellingthon_lessa,2021-01-08 17:24:27+00:00,-0.0477,neutral,"['desfile', 'em', 'e', 'que', 'duas']","['prada', 'style', 'fashion', 'fw', 'vogue', '..."
5318,816,manzanidap,2021-01-08 18:20:58+00:00,-0.0225,neutral,"['esta', 'es']","['prada', 'style', 'fashion', 'fw', 'vogue', '..."


In [24]:
import ast
def build_interaction_sequence(df):
    return df[['post_id', 'timestamp', 'polarity']].to_dict('records')

df_sequence = user_interactions.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(
    columns={0: 'interaction_sequence'}
)

df_sequence["posts_sequence"] = df_sequence["interaction_sequence"].apply(lambda x: [item["post_id"] for item in x])
df_sequence['posts_sequence'] = df_sequence['posts_sequence'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


df_sequence = df_sequence[df_sequence['posts_sequence'].apply(lambda x: len(x) > 2)]
print(f"df_sequence shape: {df_sequence.shape}")
df_sequence.head(5)

df_sequence shape: (238, 3)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23428\1532296604.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sequence = user_interactions.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(


,commentUser,interaction_sequence,posts_sequence
30,92thestudio,"[{'post_id': 188, 'timestamp': 2023-05-17 14:5...","[188, 216, 91, 57]"
108,_surisingh,"[{'post_id': 369, 'timestamp': 2023-05-18 09:0...","[369, 361, 360]"
112,_vanessa.riedl,"[{'post_id': 246, 'timestamp': 2023-05-01 13:0...","[246, 88, 57]"
116,_yycc99,"[{'post_id': 69, 'timestamp': 2023-03-08 20:40...","[69, 64, 17]"
149,adrianericmorales,"[{'post_id': 795, 'timestamp': 2023-02-23 11:5...","[795, 797, 792]"


In [13]:
df_user_profile = user_interactions.merge(df_sequence, on='commentUser', how='left')
# Merge image Description dataframe
df_user_profile = df_user_profile.merge(
    image_description[['post_id', 'description','color','category','style','occasion','keywords']], 
    on='post_id', 
    how='left'
)
# Drop keywords_comments column
df_user_profile = df_user_profile.drop(columns=['keywords_comments'])

#Short the columns based on post_id
df_user_profile = df_user_profile.sort_values(by=['post_id'])
df_user_profile.head()

,post_id,commentUser,timestamp,polarity,sentiment,keywords_posts,interaction_sequence,description,color,category,style,occasion,keywords
4547,1,wilsonjunior5055,2023-06-05 12:27:24+00:00,0.2074,neutral,"['cannes', 'dress', 'kiliancannes', 'wearing',...","[{'post_id': 14, 'timestamp': 2023-05-31 16:42...",a white dress with a white blouse and white pants,white,"blouse, dress, pants",business,work,"white, blouse, dress, pants, business, work"
4412,1,sevdalaland,2023-06-04 08:39:28+00:00,0.0278,neutral,"['cannes', 'dress', 'kiliancannes', 'wearing',...","[{'post_id': 1, 'timestamp': 2023-06-04 08:39:...",a white dress with a white blouse and white pants,white,"blouse, dress, pants",business,work,"white, blouse, dress, pants, business, work"
4372,1,grace.hanna92,2023-06-03 22:47:32+00:00,0.9732,positive,"['cannes', 'dress', 'kiliancannes', 'wearing',...","[{'post_id': 1, 'timestamp': 2023-06-03 22:47:...",a white dress with a white blouse and white pants,white,"blouse, dress, pants",business,work,"white, blouse, dress, pants, business, work"
4371,1,vivi__1988,2023-06-03 22:47:21+00:00,0.9289,positive,"['cannes', 'dress', 'kiliancannes', 'wearing',...","[{'post_id': 137, 'timestamp': 2023-05-03 20:5...",a white dress with a white blouse and white pants,white,"blouse, dress, pants",business,work,"white, blouse, dress, pants, business, work"
4456,1,stylemesoftly_,2023-06-04 15:44:54+00:00,0.2744,neutral,"['cannes', 'dress', 'kiliancannes', 'wearing',...","[{'post_id': 212, 'timestamp': 2023-05-21 13:5...",a white dress with a white blouse and white pants,white,"blouse, dress, pants",business,work,"white, blouse, dress, pants, business, work"


In [14]:
df_user_profile.to_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\user_profile.csv', index=False)